# Steering All-Traces Export Analysis

First-pass analysis notebook for the all-traces steering export.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

## Configuration

`RUN_DIR` should point at a steering export directory containing:

- `steering_baseline.parquet`
- `steering_results.parquet`
- `steering_direction_metadata.parquet`
- `run_config.json`

In [ ]:
LETTERS = ["A", "B", "C", "D", "E"]
RUN_DIR = Path("..") / ".." / "data" / "20260501-140929_Qwen-Qwen2.5-3B-Instruct_csqa_steering_all_traces"

In [ ]:
baseline_df = pd.read_parquet(RUN_DIR / "steering_baseline.parquet")
results_df = pd.read_parquet(RUN_DIR / "steering_results.parquet")
direction_df = pd.read_parquet(RUN_DIR / "steering_direction_metadata.parquet")

with open(RUN_DIR / "run_config.json", "r", encoding="utf-8") as f:
    run_config = json.load(f)

## Derived Metrics

Summary metrics are reconstructed from the stored logits and best non-choice logits.

In [ ]:
def logits_to_probability_frame(frame, prefix):
    logits = frame[[f"{prefix}logit_A", f"{prefix}logit_B", f"{prefix}logit_C", f"{prefix}logit_D", f"{prefix}logit_E"]].to_numpy(dtype=np.float64)
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    return logits, probs


def normalized_entropy(probabilities):
    entropy = -(probabilities * np.log(np.clip(probabilities, 1e-12, None))).sum(axis=1)
    return entropy / np.log(probabilities.shape[1])


def top1_top2_gap(logits):
    sorted_logits = np.sort(logits, axis=1)
    return sorted_logits[:, -1] - sorted_logits[:, -2]


clean_logits, clean_probs = logits_to_probability_frame(baseline_df, "clean_")
baseline_df["clean_answer_choice_entropy_normalized"] = normalized_entropy(clean_probs)
baseline_df["clean_answer_choice_top1_top2_logit_gap"] = top1_top2_gap(clean_logits)
baseline_df["clean_best_choice_logit"] = clean_logits.max(axis=1)
baseline_df["clean_best_choice_minus_best_non_choice_logit"] = (
    baseline_df["clean_best_choice_logit"] - baseline_df["clean_best_non_choice_logit"]
)

steered_logits, steered_probs = logits_to_probability_frame(results_df, "steered_")
results_df["steered_answer_choice_entropy_normalized"] = normalized_entropy(steered_probs)
results_df["steered_answer_choice_top1_top2_logit_gap"] = top1_top2_gap(steered_logits)
results_df["steered_best_choice_logit"] = steered_logits.max(axis=1)
results_df["steered_best_choice_minus_best_non_choice_logit"] = (
    results_df["steered_best_choice_logit"] - results_df["steered_best_non_choice_logit"]
)

In [ ]:
analysis_df = results_df.merge(
    baseline_df,
    on="example_id",
    how="left",
    validate="many_to_one",
).copy()

analysis_df["delta_answer_choice_entropy_normalized"] = (
    analysis_df["steered_answer_choice_entropy_normalized"] - analysis_df["clean_answer_choice_entropy_normalized"]
)
analysis_df["delta_answer_choice_top1_top2_logit_gap"] = (
    analysis_df["steered_answer_choice_top1_top2_logit_gap"] - analysis_df["clean_answer_choice_top1_top2_logit_gap"]
)
analysis_df["delta_best_choice_minus_best_non_choice_logit"] = (
    analysis_df["steered_best_choice_minus_best_non_choice_logit"] - analysis_df["clean_best_choice_minus_best_non_choice_logit"]
)
analysis_df["trace_bucket"] = np.where(
    analysis_df["clean_is_correct"],
    "originally_correct",
    "originally_incorrect",
)

method_order = ["contrastive_mean_direction", "probe_normal_direction"]
scale_order = sorted(analysis_df["scale"].drop_duplicates().tolist())

## Run Overview

In [ ]:
overview_df = pd.DataFrame(
    [
        {
            "model_id": run_config["model_id"],
            "train_split": run_config["train_split"],
            "target_split": run_config["target_split"],
            "n_target_examples": run_config["num_target_examples"],
            "n_layers": run_config["num_layers"],
            "methods": ", ".join(run_config["methods"]),
            "scales": ", ".join(str(x) for x in run_config["steering_scales"]),
            "clean_accuracy": baseline_df["clean_is_correct"].mean(),
        }
    ]
)

display(overview_df)
display(direction_df.round(4))

## Baseline

In [ ]:
baseline_summary_df = pd.DataFrame(
    [
        {
            "n_examples": len(baseline_df),
            "n_originally_correct": int(baseline_df["clean_is_correct"].sum()),
            "n_originally_incorrect": int((~baseline_df["clean_is_correct"]).sum()),
            "clean_accuracy": baseline_df["clean_is_correct"].mean(),
            "mean_clean_entropy": baseline_df["clean_answer_choice_entropy_normalized"].mean(),
            "mean_clean_top1_top2_gap": baseline_df["clean_answer_choice_top1_top2_logit_gap"].mean(),
            "mean_clean_best_choice_minus_best_non_choice_logit": baseline_df["clean_best_choice_minus_best_non_choice_logit"].mean(),
        }
    ]
)

display(baseline_summary_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.histplot(
    data=baseline_df,
    x="clean_answer_choice_entropy_normalized",
    hue="clean_is_correct",
    bins=40,
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    ax=axes[0],
)
axes[0].set_title("Clean Choice Entropy")
axes[0].set_xlabel("Normalized entropy")

sns.histplot(
    data=baseline_df,
    x="clean_answer_choice_top1_top2_logit_gap",
    hue="clean_is_correct",
    bins=40,
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    ax=axes[1],
)
axes[1].set_title("Clean Top1-Top2 Gap")
axes[1].set_xlabel("Logit gap")

sns.histplot(
    data=baseline_df,
    x="clean_best_choice_minus_best_non_choice_logit",
    hue="clean_is_correct",
    bins=40,
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    ax=axes[2],
)
axes[2].set_title("Clean Best-Choice Vs Best-Non-Choice")
axes[2].set_xlabel("Logit gap")

plt.tight_layout()
plt.show()

## Ungated Steering Summary

In [ ]:
summary_df = (
    analysis_df.groupby(["method", "scale", "layer_number"])
    .agg(
        n_examples=("example_id", "count"),
        prediction_change_rate=("prediction_changed", "mean"),
        rescue_rate=("rescued_error", "mean"),
        harm_rate=("harmed_correct", "mean"),
        steered_accuracy=("steered_is_correct", "mean"),
        mean_delta_entropy=("delta_answer_choice_entropy_normalized", "mean"),
        mean_delta_top1_top2_gap=("delta_answer_choice_top1_top2_logit_gap", "mean"),
        mean_delta_best_choice_minus_best_non_choice_logit=("delta_best_choice_minus_best_non_choice_logit", "mean"),
    )
    .reset_index()
)
summary_df["net_rescue_minus_harm"] = summary_df["rescue_rate"] - summary_df["harm_rate"]

display(summary_df.round(4))

In [ ]:
overall_method_scale_df = (
    summary_df.groupby(["method", "scale"])[
        [
            "prediction_change_rate",
            "rescue_rate",
            "harm_rate",
            "steered_accuracy",
            "net_rescue_minus_harm",
            "mean_delta_entropy",
            "mean_delta_top1_top2_gap",
            "mean_delta_best_choice_minus_best_non_choice_logit",
        ]
    ]
    .mean()
    .reset_index()
)

display(overall_method_scale_df.round(4))

In [ ]:
top_rescue_df = summary_df.sort_values(
    ["rescue_rate", "net_rescue_minus_harm", "harm_rate"],
    ascending=[False, False, True],
).head(15)

top_net_df = summary_df.sort_values(
    ["net_rescue_minus_harm", "rescue_rate", "harm_rate"],
    ascending=[False, False, True],
).head(15)

display(top_rescue_df.round(4))
display(top_net_df.round(4))

## Layerwise Rescue And Harm

In [ ]:
metrics = [
    ("rescue_rate", "Rescue Rate"),
    ("harm_rate", "Harm Rate"),
    ("net_rescue_minus_harm", "Rescue Minus Harm"),
    ("prediction_change_rate", "Prediction Change Rate"),
]

fig, axes = plt.subplots(len(method_order), len(metrics), figsize=(5 * len(metrics), 4.5 * len(method_order)), sharex=True)
if len(method_order) == 1:
    axes = np.array([axes])

for row_idx, method in enumerate(method_order):
    part = summary_df.loc[summary_df["method"].eq(method)].sort_values(["scale", "layer_number"])
    for col_idx, (metric, title) in enumerate(metrics):
        ax = axes[row_idx, col_idx]
        for scale in scale_order:
            scale_part = part.loc[part["scale"].eq(scale)].sort_values("layer_number")
            ax.plot(
                scale_part["layer_number"],
                scale_part[metric],
                marker="o",
                linewidth=2,
                label=f"scale={scale:g}",
            )
        if metric in ["rescue_rate", "harm_rate", "net_rescue_minus_harm"]:
            ax.axhline(0.0, color="black", linewidth=1, alpha=0.6)
        ax.set_title(f"{method}: {title}")
        ax.set_xlabel("Layer")
        ax.set_ylabel(title)
        if row_idx == 0 and col_idx == 0:
            ax.legend()

plt.tight_layout()
plt.show()

## Accuracy Under Steering

In [ ]:
fig, axes = plt.subplots(1, len(method_order), figsize=(6 * len(method_order), 4.5), sharey=True)
if len(method_order) == 1:
    axes = [axes]

for ax, method in zip(axes, method_order):
    part = summary_df.loc[summary_df["method"].eq(method)].sort_values(["scale", "layer_number"])
    for scale in scale_order:
        scale_part = part.loc[part["scale"].eq(scale)].sort_values("layer_number")
        ax.plot(
            scale_part["layer_number"],
            scale_part["steered_accuracy"],
            marker="o",
            linewidth=2,
            label=f"scale={scale:g}",
        )
    ax.axhline(baseline_df["clean_is_correct"].mean(), color="black", linewidth=1, linestyle="--", alpha=0.7)
    ax.set_title(f"{method}: Steered Accuracy")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Dataset fraction")
    ax.legend()

plt.tight_layout()
plt.show()

## Originally Incorrect Traces

In [ ]:
incorrect_df = analysis_df.loc[~analysis_df["clean_is_correct"]].copy()
incorrect_summary_df = (
    incorrect_df.groupby(["method", "scale", "layer_number"])
    .agg(
        n_examples=("example_id", "count"),
        rescue_rate=("steered_is_correct", "mean"),
        prediction_change_rate=("prediction_changed", "mean"),
        mean_delta_entropy=("delta_answer_choice_entropy_normalized", "mean"),
        mean_delta_top1_top2_gap=("delta_answer_choice_top1_top2_logit_gap", "mean"),
        mean_delta_best_choice_minus_best_non_choice_logit=("delta_best_choice_minus_best_non_choice_logit", "mean"),
    )
    .reset_index()
)

display(incorrect_summary_df.round(4))

## Originally Correct Traces

In [ ]:
correct_df = analysis_df.loc[analysis_df["clean_is_correct"]].copy()
correct_summary_df = (
    correct_df.groupby(["method", "scale", "layer_number"])
    .agg(
        n_examples=("example_id", "count"),
        harm_rate=("harmed_correct", "mean"),
        prediction_change_rate=("prediction_changed", "mean"),
        retained_accuracy=("steered_is_correct", "mean"),
        mean_delta_entropy=("delta_answer_choice_entropy_normalized", "mean"),
        mean_delta_top1_top2_gap=("delta_answer_choice_top1_top2_logit_gap", "mean"),
        mean_delta_best_choice_minus_best_non_choice_logit=("delta_best_choice_minus_best_non_choice_logit", "mean"),
    )
    .reset_index()
)

display(correct_summary_df.round(4))

## Metric Changes

In [ ]:
metric_change_specs = [
    ("mean_delta_entropy", "Mean Entropy Change"),
    ("mean_delta_top1_top2_gap", "Mean Top1-Top2 Gap Change"),
    ("mean_delta_best_choice_minus_best_non_choice_logit", "Mean Best-Choice Vs Best-Non-Choice Change"),
]

fig, axes = plt.subplots(len(method_order), len(metric_change_specs), figsize=(5 * len(metric_change_specs), 4.5 * len(method_order)), sharex=True)
if len(method_order) == 1:
    axes = np.array([axes])

for row_idx, method in enumerate(method_order):
    part = summary_df.loc[summary_df["method"].eq(method)].sort_values(["scale", "layer_number"])
    for col_idx, (metric, title) in enumerate(metric_change_specs):
        ax = axes[row_idx, col_idx]
        for scale in scale_order:
            scale_part = part.loc[part["scale"].eq(scale)].sort_values("layer_number")
            ax.plot(
                scale_part["layer_number"],
                scale_part[metric],
                marker="o",
                linewidth=2,
                label=f"scale={scale:g}",
            )
        ax.axhline(0.0, color="black", linewidth=1, alpha=0.6)
        ax.set_title(f"{method}: {title}")
        ax.set_xlabel("Layer")
        ax.set_ylabel(title)
        if row_idx == 0 and col_idx == 0:
            ax.legend()

plt.tight_layout()
plt.show()